In [1]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import pandas as pd

from config import TARGET_COLUMN

from src.data.data_loader import (
    load_supervised_data,
    load_unsupervised_data
)

from src.eda.eda_utils import (
    get_dataframe_overview,
    get_missing_summary,
    get_constant_columns,
    get_duplicate_summary,
    get_target_summary
)

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

Action Plan for Preprocessing Step:

- dropping duplicated rows
- dropping *State_ID* columns -> missing ratio = 1 
- dropping *Employee_code_ID* and *Mobile Avl Flag* columns -> nunique values is less than or equal to 1 -> non-informative for learning
- class imbalance -> minority class approx. 15% of all instances

In [2]:
### loading supervised data
# supervised dataset
df_supervised = load_supervised_data()
print(f'Supervised dataset shape: {df_supervised.shape}')

Supervised dataset shape: (59477, 23)


In [3]:
### supervised dataset - dataframe overview
overview = get_dataframe_overview(df = df_supervised)
overview

,dtype,missing_count,missing_rate,unique_count,unique_rate
State_ID,float64,59477,1.0000,0,0.0000
Employment Type,str,5851,0.0984,3,0.0001
Number of Active Accounts,float64,222,0.0037,36,0.0006
UniqueID,int64,0,0.0000,59462,0.9997
Current Balance Amount,int64,0,0.0000,39065,0.6568
Disbursed Amount,int64,0,0.0000,26668,0.4484
Sanctioned Amount,int64,0,0.0000,24892,0.4185
Instalment Amount,int64,0,0.0000,18205,0.3061
Loan To Value,float64,0,0.0000,5377,0.0904
FICO Score,int64,0,0.0000,567,0.0095


* there are some duplicated rows that are looking suspicious w.r.t. UniqueID column that we need to further investigate if we are going to drop them during preprocessing!

In [4]:
df_supervised.loc[df_supervised['UniqueID'].duplicated()] 

,UniqueID,Loan To Value,Branch ID,Age,Employment Type,DisbursalDate,State_ID,State,Employee_code_ID,Mobile Avl Flag,VoterID Flag,FICO Score,Number of Accounts,Number of Active Accounts,Number of Overdue Accounts,Current Balance Amount,Sanctioned Amount,Disbursed Amount,Instalment Amount,Number of Accounts Opened Last 6 Months,Average Account Age,Number of Inquiries,Target
18863,489281,80.26,67,57,Salaried,43347,NaN,Nevada,1998,1,0,845,1,0.0,0,0,0,0,0,0,84,0,0
18864,489281,80.26,67,57,Salaried,43347,NaN,Nevada,1998,1,0,845,1,0.0,0,0,0,0,0,0,84,0,0
18865,489281,80.26,67,57,Salaried,43347,NaN,Nevada,1998,1,0,845,1,0.0,0,0,0,0,0,0,84,0,0
18866,489281,80.26,67,57,Salaried,43347,NaN,Nevada,1998,1,0,845,1,0.0,0,0,0,0,0,0,84,0,0
18867,489281,80.26,67,57,Salaried,43347,NaN,Nevada,1998,1,0,845,1,0.0,0,0,0,0,0,0,84,0,0
31187,537409,73.23,67,33,Self employed,43369,NaN,Nevada,1998,1,0,598,1,1.0,1,27600,50200,50200,1991,0,13,0,1
31188,537409,85.00,67,41,Self employed,43399,NaN,Nevada,1998,1,1,825,1,0.0,0,0,0,0,1167,0,8,0,1
31189,537409,76.18,67,42,Self employed,43398,NaN,Nevada,1998,1,0,754,4,0.0,0,0,0,0,4719,0,12,0,0
31190,537409,58.43,67,40,Self employed,43340,NaN,Nevada,1998,1,0,554,10,3.0,1,1532662,1702000,1702000,28792,0,19,0,0
31191,537409,61.92,67,30,Self employed,43394,NaN,Nevada,1998,1,0,646,2,1.0,1,11021,49204,11021,0,1,53,0,0


* though some observations share duplicated UniqueIDs, we cannot assume that they represent the same transactions other than the ones with UniqueID of 489281; therefore we need to check duplicated rows to find out if there are some transactions that have identical records (i.e. sharing same values in all columns)

In [5]:
df_supervised.loc[df_supervised.duplicated()] 

,UniqueID,Loan To Value,Branch ID,Age,Employment Type,DisbursalDate,State_ID,State,Employee_code_ID,Mobile Avl Flag,VoterID Flag,FICO Score,Number of Accounts,Number of Active Accounts,Number of Overdue Accounts,Current Balance Amount,Sanctioned Amount,Disbursed Amount,Instalment Amount,Number of Accounts Opened Last 6 Months,Average Account Age,Number of Inquiries,Target
18863,489281,80.26,67,57,Salaried,43347,NaN,Nevada,1998,1,0,845,1,0.0,0,0,0,0,0,0,84,0,0
18864,489281,80.26,67,57,Salaried,43347,NaN,Nevada,1998,1,0,845,1,0.0,0,0,0,0,0,0,84,0,0
18865,489281,80.26,67,57,Salaried,43347,NaN,Nevada,1998,1,0,845,1,0.0,0,0,0,0,0,0,84,0,0
18866,489281,80.26,67,57,Salaried,43347,NaN,Nevada,1998,1,0,845,1,0.0,0,0,0,0,0,0,84,0,0
18867,489281,80.26,67,57,Salaried,43347,NaN,Nevada,1998,1,0,845,1,0.0,0,0,0,0,0,0,84,0,0


* we need to drop these transactions during preprocessing step

In [6]:
### supervised dataset - duplicated values
duplicated_dict = get_duplicate_summary(
    df = df_supervised,
    id_column = 'UniqueID'
)
duplicated_dict

{'n_rows': 59477, 'duplicate_rows': 5, 'duplicate_UniqueID': 15}

* summary dictionary also presents that we need to drop 4 observations from the master supervised dataset

In [7]:
### supervised dataset - missing summary
missing = get_missing_summary(df = df_supervised)
missing

,missing_count,missing_ratio
State_ID,59477,1.0000
Employment Type,5851,0.0984
Number of Active Accounts,222,0.0037


* we should be dropping State_ID columns since it only consists of missing values

In [8]:
### supervised dataset - constant columns
constant_cols = get_constant_columns(df = df_supervised)
constant_cols

['State_ID', 'Employee_code_ID', 'Mobile Avl Flag']

* these are the columns with 1 or less unique values, which will not be informative during learning; therefore we need to drop these columns before training!

In [9]:
### supervised dataset - target summary
target_summary = get_target_summary(
    df = df_supervised,
    target_column = TARGET_COLUMN
)
target_summary

,count,ratio,percentage
Target,,,
0,50482,0.848765,84.876507
1,8995,0.151235,15.123493


* we observe the class imbalance as minority class instances are composing of around 15% of all observations